# Certifying a region of attraction (Kuramoto sync)

This notebook certifies an **inner approximation of the region of attraction** of
the synchronized state of three Kuramoto oscillators, at target rate `alpha = 1`
— the paper's Fig. 3 setting — using `pyddrv.verify_roa` with the two-pass
**Trim** protocol. Requires the `[jax]` extra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyddrv import verify_roa
from pyddrv.systems.fields_jax import kuramoto_reduced_jax

## 1. The system

Three oscillators in reduced coordinates `phi_i = theta_i - theta_n` (`d = n-1 = 2`),
so the synchronized state is `phi = 0`. `kuramoto_reduced_jax` returns the batched
JAX field **and a closed-form max-norm Lipschitz bound** `L <= 2 k (n-1)/n`, so no
numerical `L` estimation is needed — we pass it directly as `L=`.

In [ ]:
k, n = 10.0, 3
f, L_bound = kuramoto_reduced_jax(k=k, n=n)
R, d = np.pi, n - 1
print("closed-form L bound:", round(L_bound, 4))

## 2. Grow the certified region

Unlike `verify_stability` (which *finds* the best rate), `verify_roa` takes a
**target rate** `alpha` and grows the set of cubes that provably converge at that
rate. `trim=True` runs the paper's two passes: pass 1 grows a tentative region
from the decay condition; pass 2 re-certifies it enforcing that trajectories stay
inside the grown region (checked against the exact cube union via a rasterized
inner-distance map).

In [ ]:
import time
t0 = time.time()
roa = verify_roa(f, R=R, d=d, alpha=1.0, L=L_bound, norm="inf",
                 tau=1.9, eps=np.pi/81, max_refine=6,
                 trim=True, raster_n=729, max_seconds=120)
print(roa.summary())
print(f"certified fraction of Q_R: {roa.volume/(2*R)**d:.1%}, "
      f"wall time {time.time()-t0:.1f}s")

The certified region should cover roughly **82% of `Q_pi`** — the synchronization
basin minus the two splay-configuration corners (which genuinely are *not* in the
basin). `roa.centers` / `roa.halfs` are the certified cubes; `roa.volume` their
total volume.

## 3. Visualize the certified region

`plot_roa_2d` draws the certified cube union, colored by cube width — you can read
the layered grid directly: coarse cubes fill the interior, geometrically finer
cubes resolve the curved boundary. The small hole at the origin is the excluded
ball `B_eps`.

In [ ]:
from pyddrv.viz import plot_roa_2d

ax = plot_roa_2d(roa, color_by="width")           # color by cube width
ax.set_xlabel(r"$\phi_1$"); ax.set_ylabel(r"$\phi_2$")
ax.set_title(f"Kuramoto (k={k:g}, n={n}): certified 1-RoA of sync")
plt.show()

`plot_roa_2d` also colors by **split depth** (`color_by="depth"`) — the
refinement level of each cube, i.e. how many `3^d` splits it took to certify.
The boundary cubes, which needed the most refinement, stand out.

In [ ]:
ax = plot_roa_2d(roa, color_by="depth")
ax.set_xlabel(r"$\phi_1$"); ax.set_ylabel(r"$\phi_2$")
ax.set_title("Certified 1-RoA, colored by split depth")
plt.show()

## Takeaways

- `verify_roa` certifies a **region**, not just a rate: every cube in `roa.centers`
  provably converges at `alpha`.
- Recurrence permits excursions that invariance-based methods (e.g. SOS) forbid,
  so the certified basin is large.
- For `d > 2`, `verify_roa` still runs; use `plot_roa_2d` on a 2-D slice, or read
  `roa.volume` directly.